# CCM-109 - Tópicos especiais de IA - Deep Learning

## Pipeline de detecção de deepfake

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jtlimo/ccm-109/blob/main/prediction.ipynb)

---

### Índice

1. [Configuração](#1-configuração)
2. [Dataset FF++](#2-dataset-faceforensics)
3. [Funções auxiliares](#3-funções-auxiliares)
4. [Treino](#4-treino)

## 1.Configuração

In [ ]:
# %pip install --upgrade pip
%pip install -q tensorflow scikit-learn keras-hub

In [2]:
# @title Configurações

import os
import glob
import keras_hub
import preprocess as pi
from pathlib import Path
import numpy as np
import cv2
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import preprocess as pi
import grad_cam as gc
import config as cfg

cfg.choice_config()

2026-08-15 17:06:16.969992: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


⚠️ Nenhum modelo encontrado! Verifique se o treinamento foi executado.
CONFIGURAÇÕES


Dropdown(description='Modelo:', layout=Layout(width='400px'), options=('model.00-0.0000.keras',), style=Descri…

Dropdown(description='Dataset:', index=3, layout=Layout(width='400px'), options=('Celeb-DF', 'DeeperForensics'…

Dropdown(description='Backbone:', layout=Layout(width='400px'), options=('Original (congelado)', 'Fine-tuned')…

FloatSlider(value=0.5, description='Threshold:', layout=Layout(width='400px'), max=0.9, min=0.1, step=0.05, st…

IntSlider(value=32, description='Batch size:', layout=Layout(width='400px'), max=128, min=8, step=8, style=Sli…

Dropdown(description='Agregação:', layout=Layout(width='400px'), options=('median', 'mean', 'max', 'trimmed_me…

IntSlider(value=10, description='Trim %:', layout=Layout(width='400px'), max=50, step=5, style=SliderStyle(des…

In [ ]:
# @title Desativar GPU

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Desativa GPU

import tensorflow as tf
from tensorflow import keras

AUTOTUNE = tf.data.AUTOTUNE

print(f"GPUs visíveis: {tf.config.list_physical_devices('GPU')}")
print(f'TensorFlow: {tf.__version__}')

## 2. Funções auxiliares

In [3]:
CFG = cfg.get_config()

print("\n" + "=" * 60)
print("CFG ATUAL:")
print("=" * 60)
for k, v in CFG.items():
    print(f"  {k:20s}: {v}")


CFG ATUAL:
  model_path          : model.00-0.0000.keras
  dataset             : Custom
  use_finetuned       : False
  threshold           : 0.5
  batch_size          : 32
  aggregation         : median
  trim_percent        : 10
  img_size            : 224
  confidence          : 0.9
  min_face_size       : 80
  skip_frames         : 9
  max_frames          : 30
  max_read_limit      : 1000
  jpg_quality         : 95
  video_extensions    : ('*.mp4', '*.avi', '*.mov', '*.mkv', '*.webm')
  siglip_mean         : [0.5 0.5 0.5]
  siglip_std          : [0.5 0.5 0.5]
  dataset_path        : /dataset/Custom
  pred_output_dir     : /dataset/Custom/predictions
  frames_output_dir   : /dataset/Custom/frames
  fake_dir            : /dataset/Custom/fake
  real_dir            : /dataset/Custom/real


In [ ]:
# @title Carregar os frames originais

# def load_original_frame(jpg_path, target_size=224):
#     img = cv2.imread(str(jpg_path))
#     img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#     img = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_LANCZOS4)
#     return img.astype(np.float32) / 255.0


# def load_original_frames(frame_paths, target_size=224):
#     frames = [load_original_frame(p, target_size) for p in frame_paths]
#     return np.stack(frames, axis=0)

In [5]:
# @title Métricas

def aggregate_mean(preds):
    return float(np.mean(preds))


def aggregate_median(preds):
    return float(np.median(preds))


def aggregate_vote(preds, threshold=0.5):
    return float(np.mean(preds > threshold))


def aggregate_trimmed_mean(preds, trim_percent=10):
    lower = np.percentile(preds, trim_percent)
    upper = np.percentile(preds, 100 - trim_percent)
    trimmed = preds[(preds >= lower) & (preds <= upper)]
    return float(np.mean(trimmed))


def aggregate_video(preds, method="median", threshold=0.5, trim_percent=10):
    if method == "mean":
        return aggregate_mean(preds)
    elif method == "median":
        return aggregate_median(preds)
    elif method == "vote":
        return aggregate_vote(preds, threshold)
    elif method == "trimmed_mean":
        return aggregate_trimmed_mean(preds, trim_percent)
    else:
        raise ValueError(f"Método desconhecido: {method}")


def classify_video(video_score, threshold=0.5):
    return "fake" if video_score > threshold else "real"

# 3. Pré-processamento

In [6]:
# @title Pre-processamento de video para predição

FEATURES_CACHE = os.path.join(CFG['dataset_path'], "siglip2_features.npz")

def load_preprocessed_frames(video_name, frames_dir, img_size=224):
    frame_dir = Path(frames_dir) / video_name
    jpg_files = sorted(frame_dir.glob("*.jpg"))
    
    if not jpg_files:
        return None
    
    frames = []
    for f in jpg_files:
        img = cv2.imread(str(f))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        normalized = pi.normalize_siglip2(img)
        frames.append(normalized)
    
    return np.stack(frames, axis=0)


def preprocess_video_for_prediction(video_path, output_dir=None):
    output_dir = output_dir or CFG["pred_output_dir"]
    video_path = Path(video_path)
    video_name = video_path.stem

    n_faces = pi.process_video(video_path, output_dir)

    if n_faces == 0:
        print(f"[AVISO] Nenhuma face detectada em {video_path.name}")
        return {"tensor": None, "frame_paths": [], "video_name": video_name}

    frames = load_preprocessed_frames(video_name, output_dir)
    video_dir = Path(output_dir) / video_name
    frame_paths = sorted(video_dir.glob("*.jpg"))


    if frames is None or len(frames) == 0:
         return {"tensor": None, "frame_paths": [], "video_name": video_name}

    return {
        "tensor": frames,
        "frame_paths": [str(p) for p in frame_paths],
        "video_name": video_name,
        "n_frames": len(frames)
    }



## 4. Predição

In [8]:
def predict_video(model, video_path, output_dir=None, cfg=None):
    cfg = cfg or CFG
    pipeline = preprocess_video_for_prediction(video_path, output_dir)

    if pipeline["tensor"] is None:
        return {
            "video_path": str(video_path),
            "video_name": pipeline["video_name"],
            "error": "Nenhuma face detectada",
            "frame_probs": np.array([]),
            "video_score": None,
            "video_label": None,
            "n_frames": 0
        }

    frames = pipeline["tensor"]
       
    frame_probs = model.model.predict(frames, batch_size=cfg["batch_size"], verbose=1).flatten()
   
    video_score = aggregate_video(
        frame_probs,
        method=cfg["aggregation"],
        threshold=cfg["threshold"],
        trim_percent=cfg["trim_percent"]
    )

    video_label = classify_video(video_score, cfg["threshold"])

    
    return {
        "video_path": str(video_path),
        "video_name": pipeline["video_name"],
        "frame_paths": pipeline["frame_paths"],
        "frame_probs": frame_probs,
        "video_score": video_score,
        "video_label": video_label,
        "n_frames": len(frame_probs),
        "tensor": frames,
    }

In [ ]:
# @title Predição de vídeos
import grad_cam as gc

def predict_and_explain_video(detector, video_path, out_dir, n_explain_frames=6, cfg=None):
    cfg = cfg or CFG
    
    result = predict_video(detector, video_path, out_dir, cfg=cfg)
    
    if result.get("error"):
        return result
    
    print(f"\n[RESULTADO] Score: {result['video_score']:.4f} | Label: {result['video_label'].upper()}")
    
    plot_video_prediction(result)
    
    frame_probs = result["frame_probs"]
    n_frames = len(frame_probs)
    
    if n_frames == 0:
        print("[AVISO] Nenhum frame para explicar.")
        return result
    
    if result["video_label"] == "fake":
        top_indices = np.argsort(frame_probs)[-n_explain_frames:][::-1]
    else:
        top_indices = np.argsort(frame_probs)[:n_explain_frames]

    top_indices = [int(i) for i in top_indices]
    print(f"[EXPLICAÇÃO] Analisando {len(top_indices)} frames: {list(top_indices)}")
    
    frames = result["tensor"]
    selected_frames = [frames[i] for i in top_indices]
    
    explanations = []
    for idx in top_indices:
        frame = frames[idx]
        
        exp = gc.explain_video_frame(detector, frame)
        exp["frame_idx"] = int(idx)
        exp["frame_prob"] = float(frame_probs[idx])
        explanations.append(exp)

    fig = gc.plot_gradcam_explanations(
        selected_frames, 
        explanations, 
        video_label=result["video_label"], 
        figsize=(20, 8)
    )
    
    return {**result, "explanations": explanations, "fig": fig}

In [10]:
# @title Avaliação do modelo

def evaluate_predictions(results, true_labels_dict):
    y_true = []
    y_pred = []
    y_scores = []
    errors = []

    for res in results:
        name = res["video_name"]

        if res.get("error"):
            errors.append({"video": name, "error": res["error"]})
            continue

        if name not in true_labels_dict:
            errors.append({"video": name, "error": "Sem ground truth"})
            continue

        y_true.append(true_labels_dict[name])
        y_pred.append(1 if res["video_label"] == "fake" else 0)
        y_scores.append(res["video_score"])

    metrics = {
        "n_evaluated": len(y_true),
        "n_errors": len(errors),
        "accuracy": accuracy_score(y_true, y_pred) if y_true else None,
        "f1": f1_score(y_true, y_pred) if y_true else None,
    }

    if len(set(y_true)) > 1:
        metrics["auc"] = roc_auc_score(y_true, y_scores)

    return {"metrics": metrics, "errors": errors, "details": list(zip(y_true, y_pred, y_scores))}



In [11]:
# @title Visualização de resultados

def plot_video_prediction(result, cfg=None, figsize=(14, 4)):
    cfg = cfg or CFG

    if result.get("error"):
        print(f"Erro: {result['error']}")
        return

    probs = result["frame_probs"]
    n = len(probs)

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    colors = ["#dc2626" if p > cfg["threshold"] else "#16a34a" for p in probs]
    axes[0].bar(range(n), probs, color=colors, alpha=0.85, edgecolor="white", linewidth=0.5)
    axes[0].axhline(y=cfg["threshold"], color="#f59e0b", linestyle="--", linewidth=2, label=f"threshold={cfg['threshold']}")
    axes[0].axhline(y=result["video_score"], color="#3b82f6", linestyle="-", linewidth=2.5, label=f"{cfg['aggregation']}={result['video_score']:.3f}")
    axes[0].set_xlabel("Frame")
    axes[0].set_ylabel("Probabilidade FAKE")
    axes[0].set_title(f"Predições por Frame | {result['video_label'].upper()}")
    axes[0].legend(fontsize=9)
    axes[0].set_ylim(0, 1)

    axes[1].hist(probs, bins=min(20, n), color="#6366f1", alpha=0.7, edgecolor="white")
    axes[1].axvline(x=result["video_score"], color="#3b82f6", linestyle="-", linewidth=2.5)
    axes[1].axvline(x=cfg["threshold"], color="#f59e0b", linestyle="--", linewidth=2)
    axes[1].set_xlabel("Probabilidade FAKE")
    axes[1].set_ylabel("Frequência")
    axes[1].set_title("Distribuição")

    axes[2].plot(range(n), probs, color="#8b5cf6", linewidth=1.5, alpha=0.8)
    axes[2].axhline(y=cfg["threshold"], color="#f59e0b", linestyle="--", linewidth=2)
    axes[2].axhline(y=result["video_score"], color="#3b82f6", linestyle="-", linewidth=2.5)
    axes[2].fill_between(range(n), 0, probs, where=(probs > cfg["threshold"]), alpha=0.2, color="#dc2626")
    axes[2].fill_between(range(n), 0, probs, where=(probs <= cfg["threshold"]), alpha=0.2, color="#16a34a")
    axes[2].set_xlabel("Frame (tempo)")
    axes[2].set_ylabel("Probabilidade FAKE")
    axes[2].set_title("Evolução Temporal")
    axes[2].set_ylim(0, 1)

    fig.suptitle(f"{result['video_name']} | Frames: {n}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


def plot_aggregation_comparison(probs, threshold=0.5, trim_percent=10):
    methods = ["mean", "median", "vote", "trimmed_mean"]
    scores = [aggregate_video(probs, m, threshold, trim_percent) for m in methods]
    labels = [classify_video(s, threshold) for s in scores]
    colors = ["#dc2626" if l == "fake" else "#16a34a" for l in labels]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(methods, scores, color=colors, alpha=0.85, edgecolor="white", linewidth=2)

    ax.axhline(y=threshold, color="#f59e0b", linestyle="--", linewidth=2)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score Agregado")
    ax.set_title("Comparação de Métodos de Agregação", pad=20)

    for bar, score, label in zip(bars, scores, labels):
        y_pos = score - 0.08 if score > 0.85 else score + 0.03
        va_align = "top" if score > 0.85 else "bottom"
        color_text = "white" if score > 0.85 else "black"


        ax.text(bar.get_x() + bar.get_width()/2, y_pos,
                f"{score:.3f}\n({label.upper()})", ha="center", va=va_align,
                fontsize=11, fontweight="bold", color= color_text)

    plt.tight_layout()
    plt.show()


def plot_dataset_results(results, cfg=None, figsize=(12, 6)):
    cfg = cfg or CFG
    valid = [r for r in results if not r.get("error")]
    if not valid:
        print("Nenhum resultado válido.")
        return

    names = [r["video_name"][:22] for r in valid]
    scores = [r["video_score"] for r in valid]
    labels = [r["video_label"] for r in valid]
    colors = ["#dc2626" if l == "fake" else "#16a34a" for l in labels]

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.barh(range(len(names)), scores, color=colors, alpha=0.85, edgecolor="white")
    ax.axvline(x=cfg["threshold"], color="#f59e0b", linestyle="--", linewidth=2)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel("Score Agregado")
    ax.set_title(f"Resultados por Vídeo | Agregação: {cfg['aggregation']}")
    ax.set_xlim(0, 1)
    ax.invert_yaxis()

    for bar, score in zip(bars, scores):
        ax.text(score + 0.02, bar.get_y() + bar.get_height()/2, f"{score:.3f}",
                va="center", fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.show()

## 5. Execução da pipeline

In [12]:
# @title Videos encontrados
os.makedirs(CFG['pred_output_dir'], exist_ok=True)

video_dir = CFG['dataset_path']
print(f"Procurando vídeos em: {video_dir}")

video_paths = sorted(glob.glob(os.path.join(video_dir, '**/*.mp4'), recursive=True))
print(f"✓ Vídeos encontrados: {len(video_paths)}")

Procurando vídeos em: /dataset/Custom
✓ Vídeos encontrados: 16


In [ ]:
# @title Carregar modelo E2E
import model as m

backbone_path = ("siglip2_base_patch16_224_finetune_backbone.keras"
if CFG['use_finetuned']
else "siglip2_base_patch16_224_original_backbone.keras"
)

if os.path.exists(backbone_path):
    detector = m.DeepfakeDetector(backbone_path, CFG['model_path'])
else:
    print(f"⚠️  Backbone não encontrado, baixando...")
    backbone = keras_hub.models.SigLIPBackbone.from_preset("siglip2_base_patch16_224")
    backbone.trainable = False
    backbone.save(backbone_path)

In [ ]:
#@title Predição de um vídeo e plot das métricas

import random

# choose random video to predict and explain
video_path = random.choice(video_paths)

print(f"Video choosed for prediction: {video_path}")


result = predict_and_explain_video(detector, video_path, CFG['pred_output_dir'])
plot_aggregation_comparison(result["probs"])